# 05 · Grounded answers with citations
Question → retrieval (best configuration from 04) → abstain check → LLM answer that cites `[S#]` → citations checked in code → support check.

- Passages are listed to the LLM as `[S1]..[Sn]`; it must cite them after each claim. Page numbers always come from chunk metadata, never from the LLM.
- Missing or invalid citations: the top retrieved chunks are shown as sources instead, and the answer is flagged.
- **Abstain** with "Your materials don't cover this." when the LLM says the passages are insufficient. A low dense-similarity floor skips the LLM call only for clearly unrelated questions.
  The first version abstained on the reranker score from 04; on reworded questions it refused 14 of 22 answerable ones, so the reranker is no longer in the answer path (it is still computed in the evaluation, for comparison).
- **Support check:** word overlap between each sentence and the passages it cites; weak sentences are flagged.
- Answer styles: concise, simple, step-by-step, worked example. At most the last 4 chat messages are sent as history.

Inputs: chunks and indexes (02, 03), `data/eval/retrieval_results.json` and `abstain_threshold.json` (04). Output: `data/eval/generation_results.jsonl` and `generation_summary.json`.

## 1 · Imports and settings

In [19]:
import csv, json, math, pickle, re, time
from collections import Counter, defaultdict
from pathlib import Path

import chromadb
import numpy as np
import requests
from sentence_transformers import CrossEncoder

DATA_DIR = Path("../data")
CHUNKS_FILE = DATA_DIR / "chunks" / "chunks.jsonl"
CHUNK_SETTINGS_FILE = DATA_DIR / "chunks" / "settings.json"
CHROMA_DIR = DATA_DIR / "chroma"
BM25_FILE = DATA_DIR / "bm25" / "bm25.pkl"
EVAL_DIR = DATA_DIR / "eval"

OLLAMA_URL = "http://localhost:11434"
LLM = "qwen2.5:7b-instruct"
NUM_CTX = 8192                 # always set: Ollama silently truncates to its default otherwise
TEMPERATURE = 0.2
K = 5
CANDIDATES = 20
RRF_K = 60
MAX_HISTORY_MESSAGES = 4
SUPPORT_MIN = 0.35             # sentence words found in its cited passages below this -> weak
INSUFFICIENT = "INSUFFICIENT_CONTEXT"
ABSTAIN_MSG = "Your materials don't cover this."
EVIDENCE_ROLES = ["textbook", "note", "web"]
RERANKER = "BAAI/bge-reranker-base"   # evaluation comparison only
DENSE_FLOOR = 0.45             # top dense cosine below this -> skip the LLM. Lowest answerable question scored 0.524 (bge-m3)

QWEN_INSTRUCT = "Instruct: Given a student's question, retrieve textbook passages that answer it\nQuery: "
EMBEDDERS = {
    "qwen3-embedding:0.6b": dict(query_prefix=QWEN_INSTRUCT, doc_prefix=""),
    "bge-m3":               dict(query_prefix="", doc_prefix=""),
    "nomic-embed-text":     dict(query_prefix="search_query: ", doc_prefix="search_document: "),
}

## 2 · Load chunks, indexes and the choices made in 04

In [20]:
def parse_config(name):
    """'dense+rerank | bge-m3' -> ('dense', True, 'bge-m3'); 'bm25' -> ('bm25', False, None)."""
    head, _, model = name.partition(" | ")
    return head.replace("+rerank", ""), "+rerank" in head, model or None


chunks = [json.loads(l) for l in CHUNKS_FILE.read_text(encoding="utf-8").splitlines()]
by_id = {c["chunk_id"]: c for c in chunks}
chunk_settings = json.loads(CHUNK_SETTINGS_FILE.read_text())
retrieval = json.loads((EVAL_DIR / "retrieval_results.json").read_text())
abstain_cfg = json.loads((EVAL_DIR / "abstain_threshold.json").read_text())
METHOD, REORDER, EMBED_MODEL = parse_config(retrieval["best"])
EMBED_MODEL = EMBED_MODEL or "nomic-embed-text"
THRESHOLD = abstain_cfg["threshold"]
assert abstain_cfg["configuration"] == retrieval["best"], "threshold was tuned on another configuration: re-run 04"

with open(BM25_FILE, "rb") as f:
    bm = pickle.load(f)
col = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection(f"chunks_{re.sub(r'[^a-z0-9]+', '_', EMBED_MODEL.lower()).strip('_')}")
assert col.metadata["chunk_settings"] == chunk_settings["hash"] == bm["chunk_settings"], "indexes are stale: re-run 03/04"
reranker = CrossEncoder(RERANKER, device="cpu")
print(f"retrieval: {METHOD} | {EMBED_MODEL} | reorder={REORDER} | dense floor {DENSE_FLOOR} | "
      f"old reranker threshold {THRESHOLD:.3f} (comparison only) | {col.count()} chunks")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 498.67it/s]


retrieval: dense | bge-m3 | reorder=False | dense floor 0.45 | old reranker threshold 0.261 (comparison only) | 495 chunks


## 3 · Retrieval (same methods as 04)

In [21]:
STOPWORDS = set("""a an the of to in on for and or is are was were be been it its this that these those with as by at from
into than then so if not no do does did can could will would should may might what which who whom how why when where
there their they them we you your our i he she his her""".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9_]+", text.lower()) if t not in STOPWORDS]


def embed_query(q):
    r = requests.post(f"{OLLAMA_URL}/api/embed", timeout=120,
                      json={"model": EMBED_MODEL, "input": [EMBEDDERS[EMBED_MODEL]["query_prefix"] + q], "keep_alive": "10m"})
    r.raise_for_status()
    v = np.asarray(r.json()["embeddings"][0], dtype=np.float32)
    return v / np.linalg.norm(v)


def where_filter(roles, sources):
    w = {"role": {"$in": list(roles)}}
    return {"$and": [w, {"source_id": {"$in": list(sources)}}]} if sources else w


def dense_search(q, n, roles, sources):
    r = col.query(query_embeddings=[embed_query(q).tolist()], n_results=n, where=where_filter(roles, sources), include=["distances"])
    return [(i, 1 - d) for i, d in zip(r["ids"][0], r["distances"][0])]


def bm25_search(q, n, roles, sources):
    scores = bm["bm25"].get_scores(tokenize(q))
    out = []
    for i in np.argsort(-scores):
        if scores[i] <= 0:
            break
        c = by_id[bm["chunk_ids"][i]]
        if c["role"] in roles and (not sources or c["source_id"] in sources):
            out.append((c["chunk_id"], float(scores[i])))
            if len(out) == n:
                break
    return out


def rrf(*lists, k=RRF_K, n=CANDIDATES):
    fused = defaultdict(float)
    for lst in lists:
        for rank, (cid, _) in enumerate(lst, start=1):
            fused[cid] += 1 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])[:n]


def retrieve(q, k=K, roles=EVIDENCE_ROLES, sources=None):
    """Return (top-k chunks, top dense cosine, candidate ids). The dense score drives the off-topic floor,
    so it is computed even when the ranking comes from BM25 or fusion."""
    dense = dense_search(q, CANDIDATES, roles, sources)
    if METHOD == "dense":
        cands = dense
    elif METHOD == "bm25":
        cands = bm25_search(q, CANDIDATES, roles, sources)
    else:
        cands = rrf(dense, bm25_search(q, CANDIDATES, roles, sources))
    ids = [cid for cid, _ in cands]
    if REORDER and ids:
        scores = rerank_scores(q, ids)
        ids = [cid for _, cid in sorted(zip(scores, ids), key=lambda x: -x[0])]
    return [by_id[cid] for cid in ids[:k]], float(dense[0][1]) if dense else 0.0, ids


def rerank_scores(q, ids):
    return [float(s) for s in reranker.predict([(q, by_id[cid]["text"]) for cid in ids], batch_size=16)]

## 4 · Prompt
The system prompt sets the citation rules and the escape word for missing information. The answer style is one extra instruction line.

In [22]:
SYSTEM_PROMPT = f"""You are a study assistant. Answer the student's question using ONLY the numbered passages from their course materials.
Rules:
- After each sentence that states a fact, cite the passage or passages it comes from, like [S1] or [S2][S3].
- Use only facts from the passages. Do not add facts from your own knowledge.
- If the passages do not contain the answer, reply with exactly {INSUFFICIENT} and nothing else.
- Cite with the passage label only, like [S2]. Never put a title or any other text inside square brackets.
- Copy code exactly as it appears in the passages, inside ``` fences.
- Write maths in plain text (for example b(new) = b(old) + a*t), not LaTeX.
- Do not mention page numbers or file names; the app shows sources."""

STYLES = {
    "concise": "Answer in 2 to 4 sentences.",
    "simple": "Explain for a beginner: short sentences, no jargon, and define any technical term you use.",
    "step-by-step": "Answer as a numbered list of steps.",
    "worked example": "Explain briefly, then walk through one worked example that uses code or numbers from the passages.",
}


def format_passages(passages):
    return "\n\n".join(f"[S{i}] ({c['source_name']}, {c['title']})\n{c['text']}" for i, c in enumerate(passages, start=1))


def build_messages(question, passages, style="concise", history=()):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    msgs += list(history)[-MAX_HISTORY_MESSAGES:]
    msgs.append({"role": "user", "content": f"Passages:\n{format_passages(passages)}\n\nStyle: {STYLES[style]}\n\nQuestion: {question}"})
    return msgs


def chat(messages, keep_alive="10m"):
    t0 = time.time()
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json={
        "model": LLM, "messages": messages, "stream": False, "keep_alive": keep_alive,
        "options": {"num_ctx": NUM_CTX, "temperature": TEMPERATURE}})
    r.raise_for_status()
    j = r.json()
    return j["message"]["content"].strip(), dict(seconds=time.time() - t0, prompt_tokens=j.get("prompt_eval_count", 0),
                                                 output_tokens=j.get("eval_count", 0))


def unload_llm():
    requests.post(f"{OLLAMA_URL}/api/chat", json={"model": LLM, "messages": [], "keep_alive": 0}, timeout=60)

## 5 · Citation parsing and validation
Accepts `[S1]`, `[S1][S2]`, `[S1, S3]` and `[S1-S3]`. Numbers outside 1..n are invalid and removed from the text. Sources shown to the student are built from the metadata of the cited chunks.

In [23]:
CITE_GROUP = re.compile(r"\[\s*S\d+(?:\s*(?:,|-|–)\s*S?\d+)*\s*\]")


def cite_numbers(group):
    nums = [int(x) for x in re.findall(r"\d+", group)]
    if re.search(r"[-–]", group) and len(nums) == 2 and nums[0] <= nums[1]:
        return list(range(nums[0], nums[1] + 1))
    return nums


def title_citations_to_labels(text, passages):
    """The model sometimes cites a passage by its title, e.g. [Tuples are more efficient]. Map those to [S#]."""
    titles = {re.sub(r"\W+", " ", c["title"]).strip().lower(): i for i, c in enumerate(passages, start=1)}
    def fix(m):
        key = re.sub(r"\W+", " ", m.group(1)).strip().lower()
        return f"[S{titles[key]}]" if key in titles else m.group(0)
    return re.sub(r"\[([^\]\[]{3,80})\]", fix, text)


def parse_citations(text, n_passages):
    valid, invalid = [], []
    def fix(m):
        nums = cite_numbers(m.group(0))
        good = [x for x in nums if 1 <= x <= n_passages]
        invalid.extend(x for x in nums if not 1 <= x <= n_passages)
        valid.extend(good)
        return "".join(f"[S{x}]" for x in good)
    cleaned = CITE_GROUP.sub(fix, text)
    return cleaned, sorted(set(valid)), invalid


def source_entry(c, n):
    pages = f"p. {c['page_start']}" + (f"-{c['page_end']}" if c["page_end"] != c["page_start"] else "")
    return dict(label=f"S{n}", source=c["source_name"], pages=pages, title=c["title"], chunk_id=c["chunk_id"])

## 6 · Support check
Every sentence outside code blocks is checked against the passages it cites: share of its content words (no stopwords, 3+ letters) found in them. Sentences below `SUPPORT_MIN`, and factual sentences with no citation, are flagged.

In [24]:
def content_words(text):
    return {t for t in tokenize(text) if len(t) > 2 and not t.isdigit()}


def split_sentences(text):
    """Split prose into sentences. Citations written after the full stop ('... plot. [S1][S2] Next ...')
    belong to the sentence before them, so they are moved back onto it."""
    prose = re.sub(r"```.*?```", " ", text, flags=re.S)
    parts = [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n+", prose) if p.strip()]
    merged = []
    for p in parts:
        lead = re.match(r"^((?:\[S\d+\]\s*)+)(.*)$", p)
        if lead and merged:
            merged[-1] += " " + lead.group(1).strip()
            p = lead.group(2).strip()
        if p:
            merged.append(p)
    return [s for s in merged if len(content_words(CITE_GROUP.sub("", s))) >= 3]


def support_check(answer, passages):
    report = []
    for s in split_sentences(answer):
        nums = [x for g in CITE_GROUP.findall(s) for x in cite_numbers(g) if 1 <= x <= len(passages)]
        words = content_words(CITE_GROUP.sub("", s))
        if not nums:
            report.append(dict(sentence=s, cited=[], overlap=0.0, flag="no citation"))
            continue
        ref = set().union(*(content_words(passages[x - 1]["text"]) for x in nums))
        overlap = len(words & ref) / len(words) if words else 1.0
        report.append(dict(sentence=s, cited=nums, overlap=round(overlap, 2), flag="weak support" if overlap < SUPPORT_MIN else None))
    return report

## 7 · The answer pipeline

In [25]:
def answer(question, style="concise", history=(), roles=EVIDENCE_ROLES, sources=None, force_llm=False):
    """force_llm=True still calls the LLM below the dense floor (only for the evaluation)."""
    t0 = time.time()
    passages, top_dense, cand_ids = retrieve(question, roles=roles, sources=sources)
    out = dict(question=question, style=style, top_dense=top_dense, retrieved=[c["chunk_id"] for c in passages],
               candidates=cand_ids, abstained=False, abstain_reason=None, answer=None, raw=None, sources=[],
               citations_ok=False, invalid_citations=[], support=[], llm=None)
    low_score = top_dense < DENSE_FLOOR or not passages
    if low_score and not force_llm:
        out.update(abstained=True, abstain_reason="off-topic (below dense floor)", answer=ABSTAIN_MSG)
    else:
        raw, stats = chat(build_messages(question, passages, style, history))
        out.update(raw=raw, llm=stats)
        if INSUFFICIENT in raw:
            out["llm_insufficient"] = True
        else:
            out["llm_insufficient"] = False
        if low_score:
            out.update(abstained=True, abstain_reason="off-topic (below dense floor)", answer=ABSTAIN_MSG)
        elif out["llm_insufficient"]:
            out.update(abstained=True, abstain_reason="model said insufficient", answer=ABSTAIN_MSG)
        else:
            text, valid, invalid = parse_citations(title_citations_to_labels(raw, passages), len(passages))
            out.update(answer=text, invalid_citations=invalid, support=support_check(text, passages))
            if valid:
                out.update(citations_ok=True, sources=[source_entry(passages[n - 1], n) for n in valid])
            else:
                out["sources"] = [source_entry(c, i) for i, c in enumerate(passages[:3], start=1)]   # fallback
    out["seconds"] = round(time.time() - t0, 2)
    return out


def show(res):
    print(f"Q: {res['question']}   [style: {res['style']}, top dense {res['top_dense']:.3f}, {res['seconds']} s]")
    print(res["answer"])
    if res["sources"]:
        tag = "Sources" if res["citations_ok"] else "Sources (fallback: no valid citations in the answer)"
        print(f"{tag}: " + "; ".join(f"[{s['label']}] {s['source']} {s['pages']} ({s['title']})" for s in res["sources"]))
    flags = [s for s in res["support"] if s["flag"]]
    for s in flags:
        print(f"  ! {s['flag']} (overlap {s['overlap']}): {s['sentence'][:110]}")
    print()

---
# Checks

## 8 · One question in all four styles

In [ ]:
for style in STYLES:
    show(answer("How does the elbow method choose the number of clusters in k-means?", style))

InternalError: Error executing plan: Internal error: Error finding id

: 

## 9 · Follow-up with chat history

In [ ]:
first = answer("What is a tuple in Python?")
show(first)
history = [{"role": "user", "content": first["question"]}, {"role": "assistant", "content": first["answer"]}]
show(answer("How is it different from a list?", history=history))
print("(the retrieval step still sees only the follow-up text; rewriting it into a standalone question is the router's job in 06)")

Q: What is a tuple in Python?   [style: concise, top dense 0.676, 5.83 s]
A tuple in Python is an ordered and fixed-length collection [S1] (immutable), which means its values cannot be changed once it is created [S3]. Tuples can contain different types of data and can be nested within each other [S1].
Sources: [S1] Ch1 and 2 Introduction to Python and Python Data Structure.pdf p. 85 (Tuple); [S3] Ch1 and 2 Introduction to Python and Python Data Structure.pdf p. 86 (Tuples are more efficient)
  ! weak support (overlap 0.11): Tuples can contain different types of data and can be nested within each other [S1].

Q: How is it different from a list?   [style: concise, top dense 0.678, 6.03 s]
A tuple in Python is an ordered and fixed-length collection (immutable), which means its values cannot be changed once it is created [S3]. Unlike lists, tuples can contain different types of data and can be nested within each other, whereas lists are mutable sequences that allow changes such as adding, 

## 10 · Citation parser on hand-made cases

In [ ]:
fake = [dict(title="Tuples are more efficient", text="")] * 1 + [dict(title="Lists", text="")]
for raw in ["Tuples are immutable [S1]. Lists are not [S2, S3].", "Range cite [S1-S3].", "Bad cite [S9] and good [S2].",
            "No citations at all.", "Fixed length [Tuples are more efficient]."]:
    print(f"{raw!r:52} ->", parse_citations(title_citations_to_labels(raw, fake), 5))
print()
for s in split_sentences("The elbow method plots SSE against k values. [S1][S2] It looks for the bend in the curve. [S3]"):
    print("sentence:", s)

'Tuples are immutable [S1]. Lists are not [S2, S3].' -> ('Tuples are immutable [S1]. Lists are not [S2][S3].', [1, 2, 3], [])
'Range cite [S1-S3].'                                -> ('Range cite [S1][S2][S3].', [1, 2, 3], [])
'Bad cite [S9] and good [S2].'                       -> ('Bad cite  and good [S2].', [2], [9])
'No citations at all.'                               -> ('No citations at all.', [], [])
'Fixed length [Tuples are more efficient].'          -> ('Fixed length [S1].', [1], [])

sentence: The elbow method plots SSE against k values. [S1][S2]
sentence: It looks for the bend in the curve. [S3]


## 11 · Full evaluation on the eval set (concise style)
The LLM is also called for questions below the dense floor (`force_llm=True`) so we can see what each abstain signal catches on its own. The pipeline answer itself is unchanged by this. The reranker score is computed after each answer, outside the timing, only for the comparison in 12b.

In [ ]:
with open(EVAL_DIR / "questions.csv", encoding="utf-8") as f:
    questions = list(csv.DictReader(f))
for q in questions:
    q["answerable"] = q["answerable"].strip().lower() == "yes"
    q["gold_pages"] = [int(p) for p in q["gold_pages"].split(";") if p.strip()]


def is_relevant(c, q):
    return c["source_name"] == q["gold_source"] and any(c["page_start"] <= p <= c["page_end"] for p in q["gold_pages"])


t0 = time.time()
runs = []
for q in questions:
    r = answer(q["question"], force_llm=True)
    r["top_rerank"] = max(rerank_scores(q["question"], r["candidates"]), default=0.0)   # comparison only, not timed
    r.update(type=q["type"], answerable=q["answerable"], difficulty=q.get("difficulty", "easy"), status=q["status"])
    r["cites_gold"] = any(is_relevant(by_id[s["chunk_id"]], q) for s in r["sources"]) if q["answerable"] and r["citations_ok"] else None
    r["retrieved_gold"] = any(is_relevant(by_id[cid], q) for cid in r["retrieved"]) if q["answerable"] else None
    runs.append(r)
print(f"{len(runs)} questions in {time.time() - t0:.0f} s")

68 questions in 710 s


## 12 · Abstain decisions

In [ ]:
def rate(xs):
    xs = list(xs)
    return f"{sum(xs)}/{len(xs)} ({sum(xs) / len(xs):.0%})" if xs else "-"


ans = [r for r in runs if r["answerable"]]
una = [r for r in runs if not r["answerable"]]
floor = lambda r: r["top_dense"] < DENSE_FLOOR
print(f"rule: abstain if LLM says insufficient; skip the LLM if top dense < {DENSE_FLOOR}")
print("answerable questions answered     :", rate(not r["abstained"] for r in ans))
print("unanswerable questions abstained  :", rate(r["abstained"] for r in una))
print("  of those, stopped by the floor   :", rate(floor(r) for r in una), "| by the LLM:", rate(r["llm_insufficient"] and not floor(r) for r in una))
print("answerable below the floor (must be 0):", sum(floor(r) for r in ans))
print()
for r in runs:
    if r["abstained"] == r["answerable"]:
        kind = "abstained on answerable" if r["answerable"] else "answered unanswerable"
        why = r["abstain_reason"] or f"dense {r['top_dense']:.3f}, LLM insufficient={r['llm_insufficient']}"
        print(f"  {kind:24} [{r['difficulty']}] ({why}) {r['question']}")

rule: abstain if LLM says insufficient; skip the LLM if top dense < 0.45
answerable questions answered     : 52/52 (100%)
unanswerable questions abstained  : 16/16 (100%)
  of those, stopped by the floor   : 1/16 (6%) | by the LLM: 15/16 (94%)
answerable below the floor (must be 0): 0



## 12a · Off-topic questions
Questions with nothing to do with the course should be stopped cheaply by the floor or by the LLM. The router in 06 will catch chit-chat before retrieval.

In [ ]:
for q in ["What is the weather today?", "Who won the football world cup?", "Tell me a joke", "What is the capital of France?"]:
    r = answer(q)
    print(f"{r['top_dense']:.3f}  {r['abstain_reason'] or 'ANSWERED'}  {r['seconds']} s  {q}")

0.525  ANSWERED  4.94 s  What is the weather today?
0.378  off-topic (below dense floor)  2.3 s  Who won the football world cup?
0.489  model said insufficient  5.19 s  Tell me a joke
0.322  off-topic (below dense floor)  2.27 s  What is the capital of France?


## 12b · Which abstain signal separates the two groups best?
Balanced accuracy of the best threshold for each signal on its own, and for "LLM says insufficient" alone (no threshold). The first-stage score is the dense cosine similarity (or the fused score for hybrid).

In [ ]:
def best_threshold(values_ans, values_una):
    best = (0, None)
    for t in sorted(set(values_ans + values_una)):
        bal = (np.mean([v >= t for v in values_ans]) + np.mean([v < t for v in values_una])) / 2
        best = max(best, (bal, t))
    return best


for key in ("top_rerank", "top_dense"):
    a, u = [r[key] for r in ans], [r[key] for r in una]
    bal, t = best_threshold(a, u)
    print(f"{key:16} best threshold {t:.3f}: balanced accuracy {bal:.2f} | answerable {np.mean([v >= t for v in a]):.0%} kept, "
          f"unanswerable {np.mean([v < t for v in u]):.0%} rejected")
llm_bal = (np.mean([not r["llm_insufficient"] for r in ans]) + np.mean([r["llm_insufficient"] for r in una])) / 2
print(f"{'LLM insufficient':16} balanced accuracy {llm_bal:.2f}")
for label, rule in [(f"old rule (rerank < {THRESHOLD:.3f} OR LLM)", lambda r: r["top_rerank"] < THRESHOLD or r["llm_insufficient"]),
                    (f"new rule (dense < {DENSE_FLOOR} OR LLM)", lambda r: r["top_dense"] < DENSE_FLOOR or r["llm_insufficient"])]:
    print(f"{label:40} balanced accuracy {(np.mean([not rule(r) for r in ans]) + np.mean([rule(r) for r in una])) / 2:.2f}")

top_rerank       best threshold 0.261: balanced accuracy 0.77 | answerable 73% kept, unanswerable 81% rejected
top_dense        best threshold 0.586: balanced accuracy 0.92 | answerable 90% kept, unanswerable 94% rejected
LLM insufficient balanced accuracy 1.00
old rule (rerank < 0.261 OR LLM)         balanced accuracy 0.87
new rule (dense < 0.45 OR LLM)           balanced accuracy 1.00


## 13 · Citations and support, on answered answerable questions

In [ ]:
answered = [r for r in ans if not r["abstained"]]
print("answers with valid citations         :", rate(r["citations_ok"] for r in answered))
print("answers with any invalid citation    :", rate(bool(r["invalid_citations"]) for r in answered))
print("answers citing a gold page           :", rate(r["cites_gold"] for r in answered if r["citations_ok"]))
print("  (gold page was in the retrieved 5  :", rate(r["retrieved_gold"] for r in answered) + ")")
sents = [s for r in answered for s in r["support"]]
print(f"sentences checked {len(sents)}: weak support {sum(s['flag'] == 'weak support' for s in sents)}, "
      f"no citation {sum(s['flag'] == 'no citation' for s in sents)}, mean overlap {np.mean([s['overlap'] for s in sents if s['cited']]):.2f}")
for d in ("easy", "hard"):
    sub = [r for r in answered if r["difficulty"] == d]
    print(f"  {d}: answered {len(sub)}, cites gold {rate(r['cites_gold'] for r in sub if r['citations_ok'])}")
print("\nflagged sentences (first 8):")
for r in answered:
    for s in r["support"]:
        if s["flag"]:
            print(f"  {s['flag']:12} {s['overlap']:.2f}  {s['sentence'][:100]}  <- {r['question'][:40]}")

answers with valid citations         : 52/52 (100%)
answers with any invalid citation    : 0/52 (0%)
answers citing a gold page           : 47/52 (90%)
  (gold page was in the retrieved 5  : 48/52 (92%))
sentences checked 102: weak support 12, no citation 22, mean overlap 0.63
  easy: answered 30, cites gold 29/30 (97%)
  hard: answered 22, cites gold 18/22 (82%)

flagged sentences (first 8):
  weak support 0.17  Tuples can contain different data types and can be nested [S1].  <- What is a tuple in Python?
  no citation  0.00  A lambda function in Python is an anonymous function created using the `lambda` keyword.  <- What is a lambda function in Python?
  no citation  0.00  The elbow method evaluates k-means clustering by calculating WCSS (Within-Cluster Sum of Square) for  <- How does the elbow method choose the num
  weak support 0.27  Pruning helps address this issue by simplifying the tree, making it less complex and more generaliza  <- Why do decision trees need pruning?
  no cit

## 14 · Speed

In [ ]:
llm_s = [r["llm"]["seconds"] for r in runs if r["llm"]]
tot = [r["seconds"] for r in runs]
print(f"LLM call: mean {np.mean(llm_s):.1f} s, p90 {np.percentile(llm_s, 90):.1f} s | whole answer: mean {np.mean(tot):.1f} s, p90 {np.percentile(tot, 90):.1f} s")
print(f"prompt tokens: mean {np.mean([r['llm']['prompt_tokens'] for r in runs if r['llm']]):.0f}, max {max(r['llm']['prompt_tokens'] for r in runs if r['llm'])} (num_ctx {NUM_CTX})")

LLM call: mean 3.4 s, p90 3.8 s | whole answer: mean 5.8 s, p90 6.2 s
prompt tokens: mean 1272, max 2244 (num_ctx 8192)


## 15 · Three answered eval questions in full

In [ ]:
for r in [x for x in answered if x["difficulty"] == "hard"][:3]:
    show(r)

Q: Why does changing one list also change another list I assigned it to?   [style: concise, top dense 0.610, 6.12 s]
When you assign a list to another variable without using a copy (e.g., `list1 = ['Computer Science', 'Computer Application', 1995, 2000]; list2 = list1`), both variables point to the same memory location. Therefore, modifying one list through these variables will affect the other because they reference the same underlying data. This behavior is due to passing by reference in Python for mutable objects like lists [S3].
Sources: [S3] Ch3 Functions Modules and Packages_part2.pdf p. 15 (passing by reference)
  ! no citation (overlap 0.0): When you assign a list to another variable without using a copy (e.g., `list1 = ['Computer Science', 'Computer
  ! no citation (overlap 0.0): Therefore, modifying one list through these variables will affect the other because they reference the same un

Q: How can I jump to a particular byte position in a file I have open?   [style: concise

## 16 · Save

In [ ]:
with open(EVAL_DIR / "generation_results.jsonl", "w", encoding="utf-8") as f:
    for r in runs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
(EVAL_DIR / "abstain_rule.json").write_text(json.dumps(dict(
    rule="abstain if the LLM replies INSUFFICIENT_CONTEXT; skip the LLM if top dense cosine < dense_floor",
    dense_floor=DENSE_FLOOR, embed_model=EMBED_MODEL, insufficient_token=INSUFFICIENT,
    lowest_answerable_dense=float(min(r["top_dense"] for r in ans))), indent=2))
summary = dict(llm=LLM, retrieval=retrieval["best"], abstain="LLM insufficient + dense floor", dense_floor=DENSE_FLOOR, n=len(runs),
               answered_answerable=float(np.mean([not r["abstained"] for r in ans])),
               abstained_unanswerable=float(np.mean([r["abstained"] for r in una])),
               citations_ok=float(np.mean([r["citations_ok"] for r in answered])),
               cites_gold=float(np.mean([r["cites_gold"] for r in answered if r["citations_ok"]])),
               mean_seconds=float(np.mean(tot)))
(EVAL_DIR / "generation_summary.json").write_text(json.dumps(summary, indent=2))
print(summary)
unload_llm()
requests.post(f"{OLLAMA_URL}/api/embed", json={"model": EMBED_MODEL, "input": [], "keep_alive": 0}, timeout=60)
print(requests.get(f"{OLLAMA_URL}/api/ps").json()["models"] or "no Ollama models loaded")

{'llm': 'qwen2.5:7b-instruct', 'retrieval': 'dense | bge-m3', 'abstain': 'LLM insufficient + dense floor', 'dense_floor': 0.45, 'n': 68, 'answered_answerable': 1.0, 'abstained_unanswerable': 1.0, 'citations_ok': 1.0, 'cites_gold': 0.9038461538461539, 'mean_seconds': 5.772647058823529}
no Ollama models loaded
